### Uncomment CKA code in src/model.py and src/train_test.py prior to CKA analysis

In [1]:
# Importing Modules
import itertools
import numpy as np
import pandas as pd
from src.dataloader import loadData
from src.model import GNNModel
from src.train_test import TrainGNN
from src.utils.smiles2morganfp import MorganFP
from src.utils.get_gnn_embedding import GetGNNEmbed
from src.utils.compute_cka import *
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/processed/train/ESOL.csv")
esol_test_data = pd.read_csv("data/processed/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/processed/train/RT.csv")
rt_test_data = pd.read_csv("data/processed/test/RT.csv")

rt_test_fp = MorganFP(rt_test_data["smiles"])
rt_test_fp["smiles"] = rt_test_fp.index
rt_test_fp = rt_test_fp.merge(rt_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/processed/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/processed/test/Lipophilicity.csv")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/processed/train/B3DB.csv")
b3db_test_data = pd.read_csv("data/processed/test/B3DB.csv")

In [3]:
# Function to run analysis
def RunGNN(train_data, test_data, dataName, modelName, params):
    
	# Training data and target labels
	smiles_X_train = train_data["smiles"]
	X_train = smiles_X_train.to_numpy()
	y_train = train_data["target"]
	y_train = y_train.to_numpy()

    # Testing data and target labels
	smiles_X_test = test_data["smiles"]
	X_test = smiles_X_test.to_numpy()
	y_test = test_data["target"]
	y_test = y_test.to_numpy()

    # Params
	h_dim, b_size, lr, d_out, wd = params

	# Loading dataset
	train_loader = loadData(X_train, y_train, batch_size=b_size)
	test_loader = loadData(X_test, y_test, batch_size=b_size, shuf=False)

	# Model
	model = GNNModel(num_features=6, hidden_dim=h_dim, model_type=modelName, dropout=d_out)

	# Model training
	model = TrainGNN(model, train_loader, epochs=100, learning_rate=lr, w_decay=wd)

	# Generate embeddings
	GNN_Embed = GetGNNEmbed(model, test_loader)

	# Return embedding array
	return GNN_Embed

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_data, "test":esol_test_data},
            "Lipophilicity":{"train":lipophilicity_train_data, "test":lipophilicity_test_data},
            "RT":{"train":rt_train_data, "test":rt_test_data},
            "B3DB":{"train":b3db_train_data, "test":b3db_test_data}
           }

# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

### CKA: GNN vs ECFP4 

In [5]:
# Storing output
temp_out = []

# Loop for models
for modelName in models:
    # Loop for dataset
    for dataName, data in datasets.items():
        # Extract params
        temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_{dataName}.csv")
        temp = temp_df[temp_df["Model"]==modelName]
        params = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay"]].to_dict('records')[0].values()
        # Run Analysis for model and dataset
        gnn = RunGNN(data["train"], data["test"], dataName, modelName, params)
        fp = MorganFP(data["test"]["smiles"]).to_numpy()
        cka = compute_cka(gnn, fp)
        temp_out.append(pd.DataFrame({ "Data": [dataName], "Model": [modelName],"CKA Score": [cka]}))

# Final output
CKA_out = pd.concat(temp_out, ignore_index=True)
CKA_out.to_csv("results/Output_CKA_Analysis.csv")
CKA_out

,Data,Model,CKA Score
0,ESOL,GCN,0.416971
1,Lipophilicity,GCN,0.284260
2,RT,GCN,0.299692
3,B3DB,GCN,0.317147
4,ESOL,GAT,0.461951
5,Lipophilicity,GAT,0.285267
6,RT,GAT,0.288624
7,B3DB,GAT,0.311683
8,ESOL,GIN,0.424562
9,Lipophilicity,GIN,0.264723


In [6]:
# Pivot to create heatmap matrix
pivot_df = CKA_out.pivot(index="Model", columns="Data", values="CKA Score")

# Plot
plt.figure(figsize=(12, 5))
sns.heatmap(pivot_df, annot=True, cmap="viridis", linewidths=0.5)
plt.xlabel("Dataset")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig(f"results/CKA_Analysis_Plot.png", dpi=300)
plt.close()

### CKA: GNN vs GNN

In [7]:
# Storing output
temp_out = []

# Generating all model combination
model_comb = list(itertools.combinations_with_replacement(models, 2))\

# Loop for models
for modelName in model_comb:
    model1 = modelName[0]
    model2 = modelName[1]
    # Loop for dataset
    for dataName, data in datasets.items():
        # Extract params
        temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_{dataName}.csv")
        temp = temp_df[temp_df["Model"]==model1]
        params1 = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay"]].to_dict('records')[0].values()
        temp = temp_df[temp_df["Model"]==model2]
        params2 = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay"]].to_dict('records')[0].values()
        # Run Analysis for model and dataset
        gnn1 = RunGNN(data["train"], data["test"], dataName, model1, params1)
        gnn2 = RunGNN(data["train"], data["test"], dataName, model2, params2)
        cka = compute_cka(gnn1, gnn2)
        temp_out.append(pd.DataFrame({ "Data": [dataName], "Model1": [model1],"Model2": [model2],"CKA Score": [cka]}))

# Final output
CKA_out = pd.concat(temp_out, ignore_index=True)
CKA_out.to_csv("results/Output_CKA_Extended_Analysis.csv")
CKA_out.head()

,Data,Model1,Model2,CKA Score
0,ESOL,GCN,GCN,0.993090
1,Lipophilicity,GCN,GCN,0.984640
2,RT,GCN,GCN,0.996711
3,B3DB,GCN,GCN,0.980124
4,ESOL,GCN,GAT,0.774587


In [8]:
datasets = CKA_out["Data"].unique()[:4] 
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 14)) 
axes_flat = axes.flatten()
sns.set_theme(style="white", context="paper")
for i, dataset_name in enumerate(datasets):
    ax = axes_flat[i]
    data = CKA_out[CKA_out["Data"] == dataset_name].reset_index(drop=True)
    models = sorted(set(data["Model1"]).union(data["Model2"]))
    mat = pd.DataFrame(np.nan, index=models, columns=models)
    for _, row in data.iterrows():
        m1, m2, v = row["Model1"], row["Model2"], row["CKA Score"]
        mat.loc[m1, m2] = v
        mat.loc[m2, m1] = v
    mat_plot = mat.loc[models[::-1], models]
    mask = np.fliplr(np.tril(np.ones_like(mat, dtype=bool), k=-1))
    sns.heatmap(
        mat_plot, 
        mask=mask, 
        annot=True, 
        fmt=".3f",
        cmap="viridis_r", 
        square=True, 
        linewidths=0.5, 
        cbar_kws={"label": "CKA Score", "shrink": 0.8}, ax=ax)
    ax.set_title(f"Dataset: {dataset_name}", fontsize=14)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig("results/CKA_Extended_Analysis_Plot.png", dpi=300, bbox_inches='tight')
plt.close()